# Vertebral Column Classification

310 patients, each described by six biomechanical features of the pelvis and spine — pelvic incidence, pelvic tilt, lumbar lordosis angle, sacral slope, pelvic radius and grade of spondylolisthesis. Predict whether the patient is normal, has a disc hernia, or has spondylolisthesis.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Vertebral Column (id 212), 3-class version
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/212/data.csv')))
print(df.shape)
print('Patients per condition:')
print(df['class'].value_counts())
df.head(3)


(310, 7)
Patients per condition:
class
Spondylolisthesis    150
Normal               100
Hernia                60
Name: count, dtype: int64


,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,class
0,63.027817,22.552586,39.609117,40.475232,98.672917,-0.254400,Hernia
1,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259,Hernia
2,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317,Hernia


In [2]:

feats = [c for c in df.columns if c != 'class']
X = df[feats]; y = df['class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.8077
Logistic Regression accuracy: 0.8077
                   precision    recall  f1-score   support

           Hernia       0.62      0.53      0.57        15
           Normal       0.72      0.72      0.72        25
Spondylolisthesis       0.93      0.97      0.95        38

         accuracy                           0.81        78
        macro avg       0.75      0.74      0.75        78
     weighted avg       0.80      0.81      0.80        78



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Which biomechanical features matter most?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


degree_spondylolisthesis    0.381677
sacral_slope                0.147542
pelvic_incidence            0.142171
pelvic_radius               0.126803
lumbar_lordosis_angle       0.125763
pelvic_tilt                 0.076044
dtype: float64


Winner: Random Forest
